In [1]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from Drag.Fuselage import Fuselage
from Drag.Bay import Bay
from Drag.LandingGear import LandingGear
from Aircraft.Aircraft import Aircraft
from global_parameters import Assumptions
from Requirements.FuelReq import FuelReq
from Requirements.LGReq import LGReq
from Requirements.MassReq import MassReq
from Requirements.MDReq import MDReq
from Requirements.EmpennageReq import EmpennageReq
from Requirements.Requirement import Requirement
from EmpennageSizing.TailFinder import TailFinder
from EmpennageSizing.CanardFinder import CanardFinder

from structural_analysis.Material import Material
from structural_analysis.iterative_planform_sizing import size_planform

In [2]:
assumptions = Assumptions()

#TODO load the fuselage here
with open("pickles/fixed_pickle.pcl", "rb") as f:
    fixed:Fixed = pickle.load(f)

for component in fixed.drag_components(False):
    component.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
    component.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
    component.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
for component in fixed.drag_components(True):
    component.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)

# Defining the standard aircraft with the standard planform
To be used when ppl don't wanna build their own planform

In [3]:
standard_wing = Planform(aspect_ratio=27, span=2.667, sweep_quarter_deg=15., taper=.5, thickness_to_chord=0.12, cm_quarter_chord=0,
                         wetted_surface_ratio=1.07, interference_factor=1.0, clmax=1.25, flap=False)

In [4]:
assumptions = Assumptions()

go_around_atmosphere = asb.Atmosphere(assumptions.altitude_go_round)
sea_level_atmosphere = asb.Atmosphere()

standard_wing.add_cache_entry('cruise', assumptions.mach_cruise, assumptions.altitude_cruise)
standard_wing.add_cache_entry('mach_max', assumptions.mach_max, assumptions.altitude_mach_max)
#NOTE: not fully technically correct but prevents coupling which would be problematic, acceptable as CD0 dept. on mach is small @ low mach
standard_wing.add_cache_entry('go_around', assumptions.airspeed_approach / go_around_atmosphere.speed_of_sound(), assumptions.altitude_go_round)
standard_wing.add_cache_entry('takeoff', assumptions.airspeed_approach / sea_level_atmosphere.speed_of_sound(), 0.)

In [5]:
material_skin = Material(assumptions.cfrp_density, elastic_modulus=assumptions.cfrp_Young_modulus, 
                         poisson_ratio=assumptions.cfrp_poisson, shear_modulus=assumptions.cfrp_Young_modulus / 2 / (1 + assumptions.cfrp_poisson),
                         yield_strength=assumptions.cfrp_yield_strength, fracture_strength=assumptions.cfrp_yield_strength)

fuselage_diameter = fixed.fuselage.diameter_max
size_planform(planform=standard_wing, thicknesses=assumptions.allowable_thicknesses, fuselage_diameter=fuselage_diameter, material_skin=material_skin, density_core=assumptions.foam_denisty)
    

Stresses 42860371.45245816, 605594930.7107136, 0.0004
Stresses 21210729.52232192, 317454172.88850045, 0.0007999999999999999
Stresses 13994182.212276518, 222023867.29797179, 0.0012
Stresses 10385908.557253804, 174797481.9375612, 0.0015999999999999999


In [6]:
print(standard_wing.mass_cache, standard_wing.x_cg_cache)

0.2976853578846578 0.2169491718718831


# We consider the thing to be tailed

In [7]:
tail = TailFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_h=7., taper_h=.7, taper_v=.8).find_planforms(standard_wing)

print(tail[0].wing_area / standard_wing.wing_area)

Stresses 26973162.73787462, 126305995.3038049, 0.0004
Stresses 13443592.632673752, 69781191.83014768, 0.0007999999999999999
Stresses 100134303.22770756, 2316834881.303295, 0.0004
Stresses 50021696.830677345, 1734672780.6129372, 0.0007999999999999999
Stresses 33317494.698333956, 1683668444.5913296, 0.0012
Stresses 24965393.632162243, 1612001707.4283195, 0.0015999999999999999
Stresses 19954132.992459252, 1300641476.879433, 0.002
Stresses 16613292.565990556, 885499809.2847117, 0.0024000000000000002
Stresses 14226977.975655785, 565874135.9629724, 0.0028
Stresses 12437242.032904698, 366996429.4207698, 0.0031999999999999997
Stresses 73128926.50839289, 322306851.9655018, 0.0004
Stresses 36446404.45206536, 166589429.31472865, 0.0007999999999999999
Stresses 24218897.099956185, 114841243.22013853, 0.0012
Stresses 18105143.423901588, 89090577.29397611, 0.0015999999999999999
Stresses 36409391.72099568, 78849372.8459209, 0.0004
Stresses 18079778.385938175, 39813535.19673669, 0.0007999999999999999
S

In [14]:
for t in tail:
    t.add_cache_entry('cruise', assumptions.mach_cruise, assumptions.altitude_cruise)
    t.add_cache_entry('mach_max', assumptions.mach_max, assumptions.altitude_mach_max)
#NOTE: not fully technically correct but prevents coupling which would be problematic, acceptable as CD0 dept. on mach is small @ low mach
    t.add_cache_entry('go_around', assumptions.airspeed_approach / go_around_atmosphere.speed_of_sound(), assumptions.altitude_go_round)
    t.add_cache_entry('takeoff', assumptions.airspeed_approach / sea_level_atmosphere.speed_of_sound(), 0.)

In [15]:
ac = Aircraft(fixed, [standard_wing] + tail)


In [16]:
for acp in ac.planforms:
    print(acp.mass_cache)

print(ac.fixed.x_cg_min, ac.fixed.x_cg_max, ac.fixed.x_LE_wing)

0.2976853578846578
0.055102418209804715
0.10860536386749538
1.0317893591765872 1.09066819465422 1.3150000000000002


# Requirement check for the aircraft

In [17]:
requirements:list[Requirement] = [
    MassReq(50.),
    MDReq(),
    FuelReq(),
    LGReq(),
    EmpennageReq(),
]

requirement_labels = [
    "MTOM",
    "Matching Diagram",
    "Fuel",
    "Landing Gear",
    "Empennage Requirement"
]

In [12]:
failed_reqs = list()
for requirement, label in zip(requirements, requirement_labels):
    if not requirement.assess(ac):
        failed_reqs.append(label)

if len(failed_reqs):
    print(f"ac mass: {ac.total_mass()}, {ac.planforms[0].oswald}")
    print(f"MainWing: AR={ac.planforms[0].aspect_ratio}, tc={ac.planforms[0].thickness_to_chord}, sweep={np.rad2deg(ac.planforms[0].sweep_quarter_rad)} deg, cmac={ac.planforms[0].cm_quarter_chord}")
    print(f"Failed: {failed_reqs}")
    print()

Fuel available: 11.000000001438 kg
Fuel required: 5.070369733990197 kg
Difference: 5.929630267447803 kg
all constraints satisfied
ac mass: 34.525554145766904, 0.680337246179267
MainWing: AR=27, tc=0.12, sweep=14.999999999999998 deg, cmac=0
Failed: ['Empennage Requirement']



In [13]:
#TODO: ctrl surface sizing